# Experiment 2 — offspring targets and online prediction

The first section inspects the fixed-reference target using three six-decision runs. The second trains the new scalar predictor online in two fourteen-decision runs, with and without using its forecasts for selection. All models start fresh. These are short implementation inspections. The final section reads the separate six-run Phase 3 comparison without launching it.

A selected parent provides one training example when its eight actual children arrive. Their novelty and comprehension are judged using the frozen selection-time value function. The retained parent is excluded; duplicate children each count. The first transition is ineligible and the last selected parent has no observed brood.

MPS is the default; change `DEVICE` to `"cpu"` without Apple GPU access. Outputs use a fresh timestamped directory and the existing experiment runner.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import replace
import json
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import display, Markdown

from automated_picbreeder.evaluation import Evaluation
from automated_picbreeder.experiment import ExperimentSettings, run_experiment
from automated_picbreeder.offspring_value import FrozenOffspringValue, reference_ranks
from automated_picbreeder.persistence import write_json
from automated_picbreeder.selection_strategies import NoveltyPredictabilitySelectionStrategy

root = Path.cwd()
if not (root / "pyproject.toml").exists():
    root = root.parent
DEVICE = "mps"
SEEDS = (7, 8, 9)
STEPS = 6
output = root / "runs" / ("offspring-target-inspection-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S-%fZ"))
output.mkdir()
print(output)


## Observe the existing policy without changing its choices

This notebook-local subclass snapshots the value function before the existing strategy trains on the current grid. At the following call it measures the actual brood. It records observations in decision metadata; it never generates alternative offspring or substitutes an imagined parent for the recorded one.

In [ ]:
class InspectTargets(NoveltyPredictabilitySelectionStrategy):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.pending = None

    def describe(self):
        return super().describe() | {"selection_strategy": "offspring-target-inspection",
                                    "policy": "unchanged novelty-predictability; no offspring predictor",
                                    "inspection_notebook": "notebooks/07_offspring_value.ipynb"}

    def choose(self, images, *, rng):
        outcome = self.pending.evaluate_offspring(images) if self.pending is not None else None
        frozen = None
        if self._previous_mean is not None:
            _, novelty, _ = self._measure(images)
            comprehension = 1 - self.observer.predict(images)[0]
            scores = ((1 - self.comprehension_weight) * reference_ranks(novelty, novelty)
                      + self.comprehension_weight * reference_ranks(comprehension, comprehension))
            position = int(scores.argmax())
            frozen = FrozenOffspringValue(reference_mean=self._previous_mean,
                novelty_reference=novelty, comprehension_reference=comprehension,
                observer=self.observer, selected_parent=images[position],
                origin_generation=self._generation, comprehension_weight=self.comprehension_weight)
            origin_values = frozen.score(images).values
        decision = super().choose(images, rng=rng)
        if frozen is not None:
            assert decision.position == position
            np.testing.assert_array_equal(origin_values[:, -1], decision.scores)
            # Even after live training, the frozen scoring function must be unchanged.
            np.testing.assert_array_equal(frozen.score(images).values, origin_values)
        self.pending = frozen
        observation = None if outcome is None else {
            "values": outcome.values.tolist(), "names": list(outcome.names), "metadata": outcome.metadata}
        evaluation = Evaluation(decision.evaluation.values, decision.evaluation.names,
            decision.evaluation.metadata | {"offspring_target_observation": observation})
        return replace(decision, evaluation=evaluation)

for seed in SEEDS:
    run_experiment(selection_strategy=InspectTargets(device=DEVICE),
                   output_dir=output / f"seed{seed}",
                   settings=ExperimentSettings(seed=seed, steps=STEPS, size=96), progress=None)
print("Diagnostic runs complete; no offspring predictor has been fitted.")


## Verify ancestry and inspect every target

The saved genome links must identify the original selected parent as the parent of all eight children. Merely looking similar is not sufficient. These are twelve factual target observations across three short runs, not evidence that they can be predicted.

In [ ]:
rows, loaded = [], {}
for seed in SEEDS:
    directory = output / f"seed{seed}"
    data = json.loads((directory / "session.json").read_text())
    events = [e for e in data["events"] if e["action"] == "select"]
    genomes = {g["key"]: g for g in data["genomes"]}
    loaded[seed] = directory, events, genomes
    for step, event in enumerate(events):
        observation = event["evaluation"]["metadata"]["offspring_target_observation"]
        if observation is None:
            assert step < 2
            continue
        meta = observation["metadata"]
        origin = events[meta["origin_generation"]]
        assert meta["origin_generation"] == step - 1
        assert event["displayed"][0] == origin["genome"]
        assert all(genomes[key]["parent"] == origin["genome"] for key in event["displayed"][1:])
        values = np.asarray(observation["values"])
        assert len(values) == 8
        ranks = values[:, 2:4]
        rows.append({"seed": seed, "origin_generation": step - 1,
                     "target": meta["mean_offspring_value"],
                     "rank_boundary_fraction": float(((ranks == 0) | (ranks == 1)).mean())})
    assert sum(e["evaluation"]["metadata"]["offspring_target_observation"] is not None for e in events) == max(STEPS - 2, 0)

targets = np.array([r["target"] for r in rows])
report = {"device": DEVICE, "steps": STEPS, "rows": rows,
          "target_min": float(targets.min()), "target_max": float(targets.max()),
          "target_std": float(targets.std()),
          "mean_rank_boundary_fraction": float(np.mean([r["rank_boundary_fraction"] for r in rows])),
          "interpretation": "Target inspection only; no forecast model trained; factual ancestry verified."}
write_json(output / "targets.json", report)
lines = ["| Seed | Origin generation (zero-based) | Mean offspring value | Rank boundary fraction |",
         "| --- | --- | --- | --- |"]
lines += [f"| {r['seed']} | {r['origin_generation']} | {r['target']:.4f} | {r['rank_boundary_fraction']:.3f} |" for r in rows]
display(Markdown("\n".join(lines)))
print({k: v for k, v in report.items() if k != "rows"})


## One predetermined transition, image by image

Inspect seed 7, generation 1 → 2 (the first valid transition). The reference mean below comes from generation 0. The top grid is the original nine-image comparison; the bottom grid contains the retained parent and its actual eight children. All displayed scores use generation 1's frozen value function.

In [ ]:
directory, events, genomes = loaded[7]
origin, arrival = events[1], events[2]
observation = arrival["evaluation"]["metadata"]["offspring_target_observation"]
def pixels(key):
    with Image.open(directory / genomes[key]["image"]) as source:
        return np.array(source.convert("RGB"))
mean = np.mean([pixels(key).astype(np.float64) / 255 for key in events[0]["displayed"]], axis=0)
reference = Image.fromarray(np.rint(mean * 255).astype(np.uint8))
reference.save(output / "reference-mean.png")
display(reference.resize((192, 192)))

canvas = Image.new("RGB", (900, 300), "white")
draw = ImageDraw.Draw(canvas)
for row, event in enumerate((origin, arrival)):
    for position, key in enumerate(event["displayed"]):
        x, y = position * 100, row * 150
        draw.text((x + 3, y + 2), f"#{key}", fill="black")
        canvas.paste(Image.fromarray(pixels(key)).resize((96, 96)), (x + 2, y + 20))
        if row == 0:
            score = origin["decision"]["scores"][position]
            label = f"value {score:.3f}" + (" *" if position == origin["position"] else "")
        else:
            label = "parent: excluded" if position == 0 else f"value {observation['values'][position - 1][-1]:.3f}"
        draw.text((x + 2, y + 119), label, fill="black")
canvas.save(output / "example-transition.png")
display(canvas)
print("* = selected parent. Mean offspring value:", observation["metadata"]["mean_offspring_value"])
columns = ["Child ID", *observation["names"]]
table = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
for key, values in zip(arrival["displayed"][1:], observation["values"], strict=True):
    table.append("| " + str(key) + " | " + " | ".join(f"{v:.6f}" for v in values) + " |")
display(Markdown("\n".join(table)))
(output / "REPORT.md").write_text("# Offspring target inspection\n\n" + report["interpretation"] +
    "\n\n" + "\n".join(lines) + "\n\n![Example transition](example-transition.png)\n\n" +
    "Generation 0 novelty reference: ![Reference mean](reference-mean.png)\n\n" +
    "\n".join(table) + f"\n\nTarget range: {targets.min():.6f}–{targets.max():.6f}; std {targets.std():.6f}.\n")
print("Report:", output / "REPORT.md")


The references are local to each selection. These values are not absolute interestingness scores. Boundary saturation can hide differences between broods. The next section tests online prediction of these targets.

## Online predictor: control and forecast-driven selection

Both conditions train a separate scalar predictor from scratch. Gamma 0 logs forecasts but uses current-image value alone; gamma 1 adds the predicted offspring value after ten valid targets have arrived. Fourteen decisions give twelve targets and three decisions with the forecast term enabled. No offspring are generated for rejected parents.

The inputs are the parent image, previous-grid mean image and 23 scoring-context values. Targets are fixed when observed; repeated parents remain separate training examples. Forecast errors below compare the original prediction with the later outcome, before training on that outcome.

In [ ]:
import time
from random import Random
from automated_picbreeder.selection_strategies import OffspringValueSelectionStrategy

ONLINE_STEPS = 14
online_settings = ExperimentSettings(seed=7, steps=ONLINE_STEPS, size=96)
online_runs = {}
for gamma in (0, 1):
    directory = output / f"online-gamma{gamma}"
    started = time.perf_counter()
    summary = run_experiment(selection_strategy=OffspringValueSelectionStrategy(gamma=gamma, device=DEVICE),
                             settings=online_settings, output_dir=directory, progress=None)
    online_runs[gamma] = {"directory": directory, "elapsed_seconds": time.perf_counter()-started}
    print(gamma, summary["decisions"], online_runs[gamma]["elapsed_seconds"])


## Original forecasts and realised outcomes

The running-mean baseline uses only targets available when the forecast was made (0.5 before any target). The other baseline predicts the parent's current value. Short-run regression loss alone cannot establish useful prediction. These observations concern selected parents only; rejected alternatives have no labels.

In [ ]:
online_rows, online_summary = [], []
for gamma, run in online_runs.items():
    data = json.loads((run["directory"] / "session.json").read_text())
    events = [e for e in data["events"] if e["action"] == "select"]
    genomes = {g["key"]: g for g in data["genomes"]}
    run.update(events=events, genomes=genomes)
    forecasts = []
    for step, event in enumerate(events):
        meta = event["evaluation"]["metadata"]["offspring"]
        feedback = meta["feedback"]
        if step:
            assert event["displayed"][0] == events[step-1]["genome"]
            assert all(genomes[k]["parent"] == events[step-1]["genome"] for k in event["displayed"][1:])
        if feedback is None:
            assert step < 2
            continue
        origin = feedback["origin_generation"]
        pending = events[origin]["evaluation"]["metadata"]["offspring"]["pending"]
        assert origin == step-1 and pending["forecast"] == feedback["forecast"]
        assert feedback["target"] == np.mean(np.asarray(feedback["child_values"])[:, -1])
        forecasts.append(feedback)
        online_rows.append({"gamma": gamma, "origin_generation": origin,
            "forecast": feedback["forecast"], "target": feedback["target"],
            "error": feedback["error"], "forecast_used": feedback["forecast_used"]})
    assert len(forecasts) == ONLINE_STEPS-2
    assert len(genomes) == 9+8*(ONLINE_STEPS-1)
    assert data["summary"]["candidate_presentations"] == 9*ONLINE_STEPS
    times = {k: sum(e["evaluation"]["metadata"]["offspring"][k] for e in events)
             for k in ("target_scoring_seconds", "predictor_training_seconds", "inference_seconds", "snapshot_seconds")}
    online_summary.append({"gamma": gamma, "targets": len(forecasts),
        "forecast_mse": float(np.mean([f["squared_error"] for f in forecasts])),
        "running_mean_mse": float(np.mean([f["running_mean_squared_error"] for f in forecasts])),
        "current_value_mse": float(np.mean([f["current_value_squared_error"] for f in forecasts])),
        "forecast_mae": float(np.mean([f["absolute_error"] for f in forecasts])),
        "target_std": float(np.std([f["target"] for f in forecasts])),
        "forecast_used_decisions": sum(e["evaluation"]["metadata"]["offspring"]["forecast_used"] for e in events),
        "changed_choices": sum(e["evaluation"]["metadata"]["offspring"]["choice_changed"] for e in events),
        "elapsed_seconds": run["elapsed_seconds"], **times})
    print(online_summary[-1])

online_table = ["| Gamma | Origin | Original forecast | Actual mean | Error | Used for choice |",
                "| --- | --- | --- | --- | --- | --- |"]
online_table += [f"| {r['gamma']} | {r['origin_generation']} | {r['forecast']:.4f} | {r['target']:.4f} | {r['error']:.4f} | {r['forecast_used']} |" for r in online_rows]
display(Markdown("\n".join(online_table)))


## Fresh replay and all final images

Replay the saved chronological grids into fresh strategy instances with the same settings and seeds. Check forecasts, targets, decisions and model hashes exactly within this device/software environment. This performs training again; it does not resume a checkpoint. Timing fields are excluded from equality.

In [ ]:
def stable(value):
    if isinstance(value, dict):
        return {k: stable(v) for k,v in value.items() if not k.endswith("_seconds")}
    if isinstance(value, list):
        return [stable(v) for v in value]
    return value

for gamma, run in online_runs.items():
    replay = OffspringValueSelectionStrategy(gamma=gamma, device=DEVICE)
    rng = Random(online_settings.resolved_selection_seed)
    for event in run["events"]:
        images = [np.array(Image.open(run["directory"] / run["genomes"][k]["image"])) for k in event["displayed"]]
        decision = replay.choose(images, rng=rng)
        assert decision.position == event["position"]
        np.testing.assert_array_equal(decision.scores, event["decision"]["scores"])
        np.testing.assert_array_equal(decision.evaluation.values, event["evaluation"]["values"])
        assert stable(decision.evaluation.metadata) == stable(event["evaluation"]["metadata"])
    # Current-image control also agrees with the independent Phase 1 inspection.
    if gamma == 0:
        baseline_events = loaded[7][1]
        for event, baseline_event in zip(run["events"], baseline_events):
            assert event["position"] == baseline_event["position"]
            assert event["evaluation"]["metadata"]["observer_state_after"] == baseline_event["evaluation"]["metadata"]["observer_state_after"]
    print("Exact replay passed:", gamma)

finals = Image.new("RGB", (400, 230), "white")
for gamma, run in online_runs.items():
    with Image.open(run["directory"] / "selected.png") as im:
        finals.paste(im.resize((192,192)), (gamma*200+4, 30))
    ImageDraw.Draw(finals).text((gamma*200+4,8), f"gamma={gamma}, final selection", fill="black")
finals.save(output / "online-final-images.png")
display(finals)
write_json(output / "online-predictions.json", {"device": DEVICE, "rows": online_rows,
           "summary": online_summary, "exact_replay_passed": True})
summary_table = ["| Gamma | Forecast MSE | Running mean MSE | Current value MSE | Choice changes | Seconds |",
                 "| --- | --- | --- | --- | --- | --- |"]
summary_table += [f"| {s['gamma']} | {s['forecast_mse']:.6f} | {s['running_mean_mse']:.6f} | {s['current_value_mse']:.6f} | {s['changed_choices']} | {s['elapsed_seconds']:.2f} |" for s in online_summary]
text = "# Online offspring prediction inspection\n\n" + "\n".join(summary_table) + "\n\n" + "\n".join(online_table)
text += "\n\n![Both final images](online-final-images.png)\n\nExact fresh replay passed for both conditions. Each run has 126 presentations, 113 genomes and 12 valid targets. The first transition is ineligible; the final selected parent has no target. Only selected-parent outcomes are observed. Fourteen decisions are too few to establish prediction quality or improved exploration.\n"
(output / "ONLINE_REPORT.md").write_text(text)
display(Markdown("\n".join(summary_table)))
print(output / "ONLINE_REPORT.md")


## Read the full Experiment 2 comparison

This section only reads and rebuilds reports. To create a new six-run batch, use `experiments/compare_offspring_value.py --device mps --output runs/my-experiment2` from the project root. Set `comparison_dir` below to the resulting directory. Every seed and endpoint is included. Prediction error is split by experience at forecast time, not by whether the forecast was used for choice.

In [ ]:
import re
from automated_picbreeder.offspring_comparison import write_offspring_report

comparison_dir = root / "runs" / "experiment2-pilot-mps"
if (comparison_dir / "comparison.json").exists():
    comparison_report = write_offspring_report(comparison_dir)
    report_text = (comparison_dir / "REPORT.md").read_text()
    report_text = re.sub(r"\]\(([^)]+)\)", lambda m: "](" + str(comparison_dir / m.group(1)) + ")", report_text)
    display(Markdown(report_text))
    display(Image.open(comparison_dir / "final-images.png"))
    for row in comparison_report["runs"]:
        print(row["condition"], row["seed"], row["status"], row.get("prediction", {}))
else:
    print("Set comparison_dir to an existing comparison; no batch was launched.")
